# 02 - CPU 工作原理（CPU Working Principle）

这个 notebook 带你**用 Python 亲手造一颗 CPU**——从寄存器到 ALU，从取指到执行，拆开来看每一个细节。

## 怎么用？
从上到下依次运行每个 cell（`Shift + Enter`），观察输出，动手改参数实验。

## 今日学习路线

| # | 内容 |
|---|------|
| 2.1 | CPU 三大组件拆解 |
| 2.2 | 寄存器组模拟 |
| 2.3 | ALU 实现 |
| 2.4 | 完整指令周期 |
| 2.5 | 时钟与频率 |
| 2.6 | 分支与跳转 |

---
## 2.1 CPU 三大组件

CPU = **控制器 (CU)** + **运算器 (ALU)** + **寄存器组 (Registers)**

```
         ┌─────────────── CPU ───────────────┐
         │                                    │
控制信号  │  ┌──────────┐    ┌──────────────┐  │
←───────┼──┤ 控制器(CU) │───→│   ALU        │  │
         │  │ 取指/译码  │    │  加减乘除/逻辑 │  │
         │  └─────┬─────┘    └──────┬───────┘  │
         │        │                 │          │
         │  ┌─────▼─────────────────▼───────┐  │
         │  │         寄存器组               │  │
         │  │  PC | IR | ACC | R0~R3 | ... │  │
         │  └──────────────────────────────┘  │
         └────────────────────────────────────┘
```

---
## 2.2 寄存器组模拟

寄存器是 CPU 内部最快的存储单元。下面我们实现一个完整的寄存器组。

In [ ]:
# ============================================
# 寄存器组实现
# ============================================

class Registers:
    """CPU 寄存器组"""
    def __init__(self):
        self.PC = 0      # Program Counter: 下一条指令地址
        self.IR = 0      # Instruction Register: 当前指令
        self.MAR = 0     # Memory Address Register
        self.MBR = 0     # Memory Buffer Register
        self.ACC = 0     # Accumulator: 累加器
        self.SP = 0xFF   # Stack Pointer: 栈指针（栈顶在内存高端）
        
        # 通用寄存器 R0 ~ R3
        self.R = [0, 0, 0, 0]
        
        # 状态标志 (Program Status Word)
        self.Z = False   # Zero: 结果为零
        self.N = False   # Negative: 结果为负
        self.C = False   # Carry: 进位/借位
        self.V = False   # Overflow: 溢出
    
    def update_flags(self, result):
        """根据 ALU 结果更新状态标志"""
        self.Z = (result == 0)
        self.N = (result < 0)
        self.V = False  # 简化：不模拟溢出
    
    def dump(self):
        """打印所有寄存器状态"""
        print(f"{'─' * 55}")
        print(f"  寄存器状态")
        print(f"{'─' * 55}")
        print(f"  PC  = {self.PC:6d} (0x{self.PC:04X})   ← 下一条指令地址")
        print(f"  IR  = {self.IR:6d} (0x{self.IR:04X})   ← 当前指令")
        print(f"  MAR = {self.MAR:6d} (0x{self.MAR:04X})   ← 内存地址寄存器")
        print(f"  MBR = {self.MBR:6d} (0x{self.MBR:04X})   ← 内存缓冲寄存器")
        print(f"  ACC = {self.ACC:6d} (0x{self.ACC:04X})   ← 累加器")
        print(f"  SP  = {self.SP:6d} (0x{self.SP:04X})   ← 栈指针")
        print(f"  R0  = {self.R[0]:6d}   R1  = {self.R[1]:6d}")
        print(f"  R2  = {self.R[2]:6d}   R3  = {self.R[3]:6d}")
        print(f"  标志: Z={self.Z} N={self.N} C={self.C} V={self.V}")
        print(f"{'─' * 55}")

reg = Registers()
reg.PC = 0x0100
reg.ACC = 42
reg.R[0] = 100
reg.dump()

---
## 2.3 ALU 实现

ALU（算术逻辑单元）是 CPU 的"计算器"。

In [ ]:
# ============================================
# ALU 实现
# ============================================

class ALU:
    """算术逻辑单元"""
    
    # 操作码定义
    ADD = 0   # 加法
    SUB = 1   # 减法
    MUL = 2   # 乘法
    AND = 3   # 按位与
    OR  = 4   # 按位或
    XOR = 5   # 按位异或
    NOT = 6   # 按位非（只取 A）
    SHL = 7   # 左移
    SHR = 8   # 右移
    CMP = 9   # 比较（A - B，只设标志不存结果）
    
    OP_NAMES = {0:'ADD', 1:'SUB', 2:'MUL', 3:'AND', 4:'OR',
                5:'XOR', 6:'NOT', 7:'SHL', 8:'SHR', 9:'CMP'}
    
    def compute(self, opcode, a, b=0):
        """
        执行 ALU 操作
        返回 (result, flags)
        flags = {'Z': zero, 'N': negative, 'C': carry, 'V': overflow}
        """
        result = 0
        carry = False
        
        if opcode == self.ADD:
            result = a + b
            carry = result > 0x7FFF  # 简化溢出检测
        elif opcode == self.SUB:
            result = a - b
        elif opcode == self.MUL:
            result = a * b
        elif opcode == self.AND:
            result = a & b
        elif opcode == self.OR:
            result = a | b
        elif opcode == self.XOR:
            result = a ^ b
        elif opcode == self.NOT:
            result = ~a & 0xFFFF  # 16-bit 取反
        elif opcode == self.SHL:
            result = (a << b) & 0xFFFF
        elif opcode == self.SHR:
            result = a >> b
        elif opcode == self.CMP:
            result = a - b  # 计算差值但不返回
        
        flags = {
            'Z': result == 0,
            'N': result < 0,
            'C': carry,
            'V': False
        }
        return result, flags

# 测试 ALU
alu = ALU()

print("🧮 ALU 运算测试\n")
print(f"{'操作':>6s} | {'A':>6s} | {'B':>6s} | {'结果':>6s} | Z | N")
print("-" * 45)

tests = [
    (ALU.ADD, 10, 20),
    (ALU.SUB, 50, 30),
    (ALU.MUL, 6, 7),
    (ALU.AND, 0b1100, 0b1010),
    (ALU.OR,  0b1100, 0b1010),
    (ALU.XOR, 0b1100, 0b1010),
    (ALU.NOT, 0b0000, 0),
    (ALU.SHL, 1, 4),
    (ALU.SHR, 128, 2),
    (ALU.CMP, 10, 5),
]

for op, a, b in tests:
    res, flags = alu.compute(op, a, b)
    op_name = ALU.OP_NAMES[op]
    if op == ALU.CMP:
        print(f"{op_name:>6s} | {a:6d} | {b:6d} | (仅设标志) | {flags['Z']!r:>5s} | {flags['N']!r:>5s}")
    else:
        print(f"{op_name:>6s} | {a:6d} | {b:6d} | {res:6d} | {flags['Z']!r:>5s} | {flags['N']!r:>5s}")

### ✏️ 二进制可视化

下面我们看到 ALU 按位运算的实际效果。

In [ ]:
def show_bitwise(a, b):
    """可视化按位逻辑运算"""
    alu = ALU()
    a_bin = f'{a:08b}'
    b_bin = f'{b:08b}'
    
    print(f"  A    = {a_bin}  ({a})")
    print(f"  B    = {b_bin}  ({b})")
    print(f"  ────────────────")
    
    for op in [ALU.AND, ALU.OR, ALU.XOR]:
        res, _ = alu.compute(op, a, b)
        name = ALU.OP_NAMES[op]
        print(f"  {name:<4s} = {res:08b}  ({res})")

show_bitwise(0b11001100, 0b10101010)
print()
show_bitwise(0b11110000, 0b00001111)

---
## 2.4 完整指令周期模拟

有了内存、寄存器、ALU，现在我们把它们串联成完整的**取指→译码→执行→写回**循环。

这次我们实现一个**更强大的 CPU**：支持 8 条指令，4 个通用寄存器。

In [ ]:
# ============================================
# 完整 CPU 模拟器
# ============================================

class Memory:
    def __init__(self, size=256):
        self.size = size
        self.cells = [0] * size
    def read(self, addr):
        return self.cells[addr] if 0 <= addr < self.size else 0
    def write(self, addr, val):
        if 0 <= addr < self.size:
            self.cells[addr] = val & 0xFFFF
    def dump(self, start, end):
        for addr in range(start, end):
            print(f"  [{addr:3d}] = {self.cells[addr]:5d} (0x{self.cells[addr]:04X})")
        print()


class CPU:
    """一个精简但完整的 CPU"""
    
    # ── 指令集定义 ──
    # 指令格式：[操作码, 操作数1, 操作数2, 操作数3]
    # 长度不等：LOAD 占 2 字，ADD/SUB/MUL 占 4 字，其他占 2 字
    
    HALT  = 0x00   # 停机
    LOAD  = 0x01   # LOAD  Rx, [addr]    → Rx = Memory[addr]
    STORE = 0x02   # STORE [addr], Rx    → Memory[addr] = Rx
    ADD   = 0x03   # ADD   Rd, Rs1, Rs2  → Rd = Rs1 + Rs2
    SUB   = 0x04   # SUB   Rd, Rs1, Rs2  → Rd = Rs1 - Rs2
    MUL   = 0x05   # MUL   Rd, Rs1, Rs2  → Rd = Rs1 * Rs2
    JMP   = 0x06   # JMP   addr          → PC = addr（无条件跳转）
    JZ    = 0x07   # JZ    addr          → 如果 Zero 标志=1，PC = addr
    CMP   = 0x08   # CMP   Rs1, Rs2      → 比较两数，设标志
    MOV   = 0x09   # MOV   Rd, Rs        → Rd = Rs（寄存器间拷贝）
    INC   = 0x0A   # INC   Rx            → Rx = Rx + 1
    
    MNEMONICS = {
        0x00: 'HALT', 0x01: 'LOAD', 0x02: 'STORE',
        0x03: 'ADD',  0x04: 'SUB',  0x05: 'MUL',
        0x06: 'JMP',  0x07: 'JZ',   0x08: 'CMP',
        0x09: 'MOV',  0x0A: 'INC'
    }
    
    def __init__(self, memory_size=256):
        self.mem = Memory(memory_size)
        self.reg = Registers()
        self.alu = ALU()
        self.running = False
        self.cycle = 0
        self.trace_log = []
    
    # ---------- 读写内存辅助 ----------
    def _read_mem(self, addr):
        """模拟内存读（经过 MAR → MBR）"""
        self.reg.MAR = addr
        self.reg.MBR = self.mem.read(self.reg.MAR)
        return self.reg.MBR
    
    def _write_mem(self, addr, val):
        """模拟内存写（经过 MAR, MBR）"""
        self.reg.MAR = addr
        self.reg.MBR = val
        self.mem.write(self.reg.MAR, self.reg.MBR)
    
    # ---------- 取指 ----------
    def fetch(self):
        """取指阶段：从内存读指令到 IR"""
        self.reg.IR = self._read_mem(self.reg.PC)
        self.reg.PC += 1
        return self.reg.IR
    
    # ---------- 译码+执行 ----------
    def decode_execute(self, opcode):
        """译码并执行当前指令"""
        
        if opcode == self.HALT:
            self.running = False
            return "HALT"
        
        elif opcode == self.LOAD:
            reg_idx = self.fetch()       # 读寄存器编号
            addr = self.fetch()           # 读内存地址
            val = self._read_mem(addr)    # 从内存读数据
            self.reg.R[reg_idx] = val
            self.reg.update_flags(val)
            return f"LOAD R{reg_idx}, [{addr}] → R{reg_idx}={val}"
        
        elif opcode == self.STORE:
            addr = self.fetch()
            reg_idx = self.fetch()
            val = self.reg.R[reg_idx]
            self._write_mem(addr, val)
            return f"STORE [{addr}], R{reg_idx} → mem[{addr}]={val}"
        
        elif opcode == self.ADD:
            rd = self.fetch()
            rs1 = self.fetch()
            rs2 = self.fetch()
            a, b = self.reg.R[rs1], self.reg.R[rs2]
            result, flags = self.alu.compute(ALU.ADD, a, b)
            self.reg.R[rd] = result
            self._update_flags_dict(flags)
            return f"ADD R{rd}, R{rs1}, R{rs2} → R{rd}={a}+{b}={result}"
        
        elif opcode == self.SUB:
            rd = self.fetch()
            rs1 = self.fetch()
            rs2 = self.fetch()
            a, b = self.reg.R[rs1], self.reg.R[rs2]
            result, flags = self.alu.compute(ALU.SUB, a, b)
            self.reg.R[rd] = result
            self._update_flags_dict(flags)
            return f"SUB R{rd}, R{rs1}, R{rs2} → R{rd}={a}-{b}={result}"
        
        elif opcode == self.MUL:
            rd = self.fetch()
            rs1 = self.fetch()
            rs2 = self.fetch()
            a, b = self.reg.R[rs1], self.reg.R[rs2]
            result, flags = self.alu.compute(ALU.MUL, a, b)
            self.reg.R[rd] = result
            self._update_flags_dict(flags)
            return f"MUL R{rd}, R{rs1}, R{rs2} → R{rd}={a}×{b}={result}"
        
        elif opcode == self.JMP:
            addr = self.fetch()
            old_pc = self.reg.PC
            self.reg.PC = addr
            return f"JMP {addr} (PC: {old_pc} → {addr})"
        
        elif opcode == self.JZ:
            addr = self.fetch()
            if self.reg.Z:
                old_pc = self.reg.PC
                self.reg.PC = addr
                return f"JZ {addr} → 跳转! (Z=1, PC→{addr})"
            else:
                return f"JZ {addr} → 不跳转 (Z=0)"
        
        elif opcode == self.CMP:
            rs1 = self.fetch()
            rs2 = self.fetch()
            a, b = self.reg.R[rs1], self.reg.R[rs2]
            _, flags = self.alu.compute(ALU.CMP, a, b)
            self._update_flags_dict(flags)
            return f"CMP R{rs1}, R{rs2} ({a} vs {b}) → Z={self.reg.Z}, N={self.reg.N}"
        
        elif opcode == self.MOV:
            rd = self.fetch()
            rs = self.fetch()
            self.reg.R[rd] = self.reg.R[rs]
            return f"MOV R{rd}, R{rs} → R{rd}={self.reg.R[rs]}"
        
        elif opcode == self.INC:
            rx = self.fetch()
            self.reg.R[rx] += 1
            self.reg.update_flags(self.reg.R[rx])
            return f"INC R{rx} → R{rx}={self.reg.R[rx]}"
        
        else:
            self.running = False
            return f"??? 未知指令 {opcode}"
    
    def _update_flags_dict(self, flags_dict):
        """从 ALU 返回的 flags 字典更新寄存器标志"""
        self.reg.Z = flags_dict.get('Z', False)
        self.reg.N = flags_dict.get('N', False)
        self.reg.C = flags_dict.get('C', False)
        self.reg.V = flags_dict.get('V', False)
    
    # ---------- 运行 ----------
    def load_program(self, program_dict, data_dict=None):
        """加载程序和数据到内存"""
        for addr, val in program_dict.items():
            self.mem.write(addr, val)
        if data_dict:
            for addr, val in data_dict.items():
                self.mem.write(addr, val)
    
    def run(self, verbose=True):
        """运行程序直到 HALT"""
        self.reg.PC = 0
        self.cycle = 0
        self.running = True
        self.trace_log = []
        
        print("╔" + "═" * 58 + "╗")
        print("║  🚀 CPU 开始执行" + " " * 42 + "║")
        print("╚" + "═" * 58 + "╝\n")
        
        while self.running and self.reg.PC < self.mem.size:
            self.cycle += 1
            
            # ── FETCH ──
            opcode = self.fetch()
            mnemonic = self.MNEMONICS.get(opcode, f'???({opcode})')
            
            # ── DECODE & EXECUTE ──
            desc = self.decode_execute(opcode)
            
            if verbose:
                pc_before = self.reg.PC  # 已经是更新后的
                print(f"  [{self.cycle:2d}] {mnemonic:<5s}  {desc}")
            
            self.trace_log.append({
                'cycle': self.cycle,
                'opcode': opcode,
                'mnemonic': mnemonic,
                'description': desc,
                'z_flag': self.reg.Z,
                'n_flag': self.reg.N
            })
            
            if self.cycle > 100:
                print("  ⚠️ 超过 100 周期，强制停止")
                break
        
        print(f"\n✅ 执行完毕，共 {self.cycle} 个周期\n")
    
    def dump_state(self):
        """打印 CPU 最终状态"""
        self.reg.dump()
        print("内存（数据区）:")
        self.mem.dump(0x50, 0x60)

### 运行第一个完整程序

计算 `(100 + 50) × 3`，把结果存到内存地址 0x80。

In [ ]:
cpu = CPU()

# 汇编伪代码：
#   LOAD R0, [0x90]    ; R0 = 100
#   LOAD R1, [0x91]    ; R1 = 50
#   ADD  R2, R0, R1    ; R2 = 100 + 50 = 150
#   LOAD R3, [0x92]    ; R3 = 3
#   MUL  R0, R2, R3    ; R0 = 150 × 3 = 450
#   STORE [0x80], R0   ; 把 450 存到内存 0x80
#   HALT

program = {
    0x00: CPU.LOAD,  0x01: 0,  0x02: 0x90,    # LOAD R0, [0x90]
    0x03: CPU.LOAD,  0x04: 1,  0x05: 0x91,    # LOAD R1, [0x91]
    0x06: CPU.ADD,   0x07: 2,  0x08: 0, 0x09: 1,  # ADD R2, R0, R1
    0x0A: CPU.LOAD,  0x0B: 3,  0x0C: 0x92,    # LOAD R3, [0x92]
    0x0D: CPU.MUL,   0x0E: 0,  0x0F: 2, 0x10: 3,  # MUL R0, R2, R3
    0x11: CPU.STORE, 0x12: 0x80, 0x13: 0,      # STORE [0x80], R0
    0x14: CPU.HALT,
}

data = {
    0x90: 100,   # 数据区
    0x91: 50,
    0x92: 3,
}

cpu.load_program(program, data)
cpu.run()

print("── 最终状态 ──")
cpu.dump_state()
print(f"\n🔍 验证：mem[0x80] = {cpu.mem.read(0x80)}  (预期: 450)")

### ✏️ 自己动手

修改程序：计算 `(200 - 80) × 2`，结果存到 0x80。

In [ ]:
# 👇 复制上面的代码并修改 program 和 data
cpu2 = CPU()

my_program = {
    # 在这里写你的程序
    # 提示：用 SUB 代替 ADD
}

my_data = {
    0x90: 200,
    0x91: 80,
    0x92: 2,
}

# cpu2.load_program(my_program, my_data)
# cpu2.run()
# print(f"\n🔍 验证：mem[0x80] = {cpu2.mem.read(0x80)}")
print("✏️ 写完后取消注释来运行")

---
## 2.5 时钟与性能

理解 CPU 时钟频率、周期、CPI（每条指令周期数）和 IPC（每周期指令数）的关系。

In [ ]:
# ============================================
# 性能模拟：频率 vs IPC
# ============================================

def cpu_performance_sim(clock_ghz, ipc, instruction_count):
    """
    根据频率和 IPC 计算执行时间
    
    参数:
        clock_ghz: 主频 (GHz)
        ipc: 每周期执行的指令数
        instruction_count: 总指令数
    
    返回:
        执行时间 (秒)
    """
    cycles = instruction_count / ipc
    cycle_time_ns = 1 / clock_ghz  # 纳秒
    total_time_ns = cycles * cycle_time_ns
    return total_time_ns * 1e-9

# 比较不同 CPU 配置
print("📊 CPU 性能对比（执行 100 亿条指令）\n")
print(f"{'配置':<45s} | {'耗时':>10s} | {'相对性能':>10s}")
print("-" * 75)

configs = [
    ("老式单核 2GHz, IPC=1", 2.0, 1.0),
    ("高频单核 5GHz, IPC=1", 5.0, 1.0),
    ("现代核 3GHz, IPC=2.5  (Apple M1 风格)", 3.0, 2.5),
    ("现代核 4GHz, IPC=4    (现代 x86 风格)", 4.0, 4.0),
]

instructions = 10e9  # 100 亿条指令
baseline_time = None

for name, freq, ipc in configs:
    elapsed = cpu_performance_sim(freq, ipc, instructions)
    if baseline_time is None:
        baseline_time = elapsed
    relative = baseline_time / elapsed
    print(f"{name:<45s} | {elapsed:8.3f}s | {relative:8.1f}x")

print(f"\n💡 关键洞察：")
print(f"   - 频率翻倍 (2→5GHz, IPC相同): 性能翻 2.5x")
print(f"   - IPC 翻倍 (1→2.5, 频率更低): 性能反而更好！")
print(f"   - 所以：IPC 和频率同样重要，甚至更重要")

In [ ]:
# ============================================
# 时钟信号可视化
# ============================================

def draw_clock_waveform(cycles=5, width=40):
    """绘制时钟方波"""
    # 每个周期：上升沿 + 下降沿
    for cycle in range(1, cycles + 1):
        # 高电平
        print(f"  ┌{'─' * (width // 2)}┐  ← 上升沿（触发动作）")
        print(f"  │{' ' * (width // 2)}│")
        print(f"  └{'─' * (width // 2)}┘  ← 下降沿")
        print(f"  ├───────── {cycle * 0.33:.2f} ns (3GHz) ────────┤\n")

print("⏱️ 时钟信号波形（3GHz，每周期 0.33ns）:\n")
draw_clock_waveform(3)

---
## 2.6 分支与循环：JMP 和 JZ

没有分支指令，程序只能一条道跑到黑。下面看 CPU 如何实现循环。

In [ ]:
# ============================================
# 程序：用循环计算 1+2+3+4+5 = 15
# ============================================

cpu3 = CPU()

# 算法：
#   R0 = 0    (累加和)
#   R1 = 5    (计数器，从 5 倒数到 0)
#   loop:
#     ADD R0, R0, R1   ; sum += counter
#     DEC R1           ; counter -= 1  (我们用 SUB R1, R1, R2, R2 存 1)
#     CMP R1, R3       ; counter == 0? (R3 存 0)
#     JZ  done         ; 如果=0，跳转到 done
#     JMP  loop        ; 否则继续循环
#   done:
#     STORE [0x80], R0 ; 存结果
#     HALT

program_loop = {
    # 初始化
    0x00: CPU.LOAD, 0x01: 0, 0x02: 0xA0,    # LOAD R0, [0xA0]  → R0=0 (sum)
    0x03: CPU.LOAD, 0x04: 1, 0x05: 0xA1,    # LOAD R1, [0xA1]  → R1=5 (counter)
    0x06: CPU.LOAD, 0x07: 2, 0x08: 0xA2,    # LOAD R2, [0xA2]  → R2=1 (用于减法)
    0x09: CPU.LOAD, 0x0A: 3, 0x0B: 0xA3,    # LOAD R3, [0xA3]  → R3=0 (用于比较)
    
    # loop: (地址 0x0C)
    0x0C: CPU.ADD,  0x0D: 0, 0x0E: 0, 0x0F: 1,   # ADD R0, R0, R1  (sum += counter)
    0x10: CPU.SUB,  0x11: 1, 0x12: 1, 0x13: 2,   # SUB R1, R1, R2  (counter -= 1)
    0x14: CPU.CMP,  0x15: 1, 0x16: 3,            # CMP R1, R3      (counter < 0?)
    0x17: CPU.JZ,   0x18: 0x1B,                   # JZ done (0x1B)  如果 counter < 0 跳转
    0x19: CPU.JMP,  0x1A: 0x0C,                   # JMP loop (0x0C)
    
    # done: (地址 0x1B)
    0x1B: CPU.STORE, 0x1C: 0x80, 0x1D: 0,        # STORE [0x80], R0
    0x1E: CPU.HALT,
}

data_loop = {
    0xA0: 0,    # sum = 0
    0xA1: 5,    # counter
    0xA2: 1,    # constant 1
    0xA3: 0,    # constant 0
}

cpu3.load_program(program_loop, data_loop)
cpu3.run()

print("── 最终状态 ──")
cpu3.dump_state()
print(f"\n🔍 验证：mem[0x80] = {cpu3.mem.read(0x80)}  (预期: 5+4+3+2+1+0 = 15)")

### 🔍 追踪循环执行过程

In [ ]:
# 用 pandas 看每一步
import pandas as pd
df = pd.DataFrame(cpu3.trace_log)
print(df[['cycle', 'mnemonic', 'description']].to_string(index=False))

### ✏️ 自己动手

修改程序，计算 10 的阶乘（10! = 10×9×8×...×1）。提示：
- R0 = 1 (乘积初始值)
- R1 = 10 (计数器，从 10 倒数到 1)
- 循环：R0 = R0 × R1, R1 = R1 - 1, 当 R1 > 0 时继续

（注意：这个 CPU 没有 JG/JL 指令，你需要比较后判断 Z 标志。提示：先 CMP 再检查——但 CMP 后 Z=1 意味着相等。如果要判断 > 0，可以：每次减 1 后，CMP R1, R3 (R3存0)，然后 JZ... 但 JZ 是等于才跳。你需要在计数器变成 0 时**不跳转**而是继续做最后一次乘法，然后下一次 CMP R1, R3 (R3存0) 时跳转。）

In [ ]:
# 👇 在这里写你的阶乘程序
cpu_fact = CPU()

# 提示：先画出流程图，再翻译成指令
# counter 从 10 减到 1，每次乘到积里
# counter = 0 时跳出循环

fact_program = {
    # 你的代码
}

fact_data = {
    0xA0: 1,   # product = 1
    0xA1: 10,  # counter
    0xA2: 1,   # constant 1
    0xA3: 0,   # constant 0
}

# cpu_fact.load_program(fact_program, fact_data)
# cpu_fact.run()
# print(f"\n🔍 mem[0x80] = {cpu_fact.mem.read(0x80)}  (预期: 3628800 = 10!)")
print("✏️ 写完后取消注释来运行")

---
## 🎯 综合挑战

In [ ]:
# Q1: 以下哪个寄存器存放「当前正在执行的指令」？
# A) PC   B) IR   C) MAR   D) MBR
q1 = ""
print(f"Q1 答案: {q1}")

In [ ]:
# Q2: 3GHz 的 CPU，一个时钟周期是多少纳秒？
q2_period_ns = 1 / 3_000_000_000  # 👈 改这里（单位：秒）
print(f"Q2 时钟周期 = {q2_period_ns * 1e9:.3f} ns")
print(f"   验证: {q2_period_ns * 1e9:.3f} ns (应该 ≈ 0.333 ns)")

In [ ]:
# Q3: 某 CPU 执行 1000 条指令用了 5000 个周期，CPI 是多少？
total_cycles = 5000
total_instructions = 1000
cpi = None  # 👈 计算
print(f"Q3 CPI = {cpi} 个周期/指令")
print(f"   IPC = {1/cpi if cpi else '?'} 条指令/周期")

---
## ✅ 检查清单

对照一下，都掌握了吗？

- [ ] CPU 三大组件：CU、ALU、寄存器组
- [ ] 关键寄存器：PC, IR, MAR, MBR, ACC, SP, FLAGS
- [ ] 指令周期四个阶段：Fetch → Decode → Execute → Writeback
- [ ] ALU 支持的运算：算术（+-×÷）、逻辑（AND/OR/NOT/XOR）、移位、比较
- [ ] 状态标志：Z(零), N(负), C(进位), V(溢出)
- [ ] 时钟频率、周期、CPI、IPC 之间的关系
- [ ] 分支/跳转指令（JMP/JZ）如何实现循环
- [ ] 能阅读和编写简单的汇编程序

---

> 📖 下一章：[03 - 指令集与汇编基础](../03-instruction-set-and-assembly/) — 深入理解 ISA、RISC vs CISC、汇编编程